# 00 — Загрузка исходных баз растворимости

Первый этап проекта. Загружает официальные CSV **MixtureSolDB** и **BigSolDB** с Zenodo,
проверяет контрольные суммы, сохраняет метаданные и пути для ноутбука `01`.
Исходные таблицы сохраняются без преобразований.

**Проверено 01.10.2026:**
- [MixtureSolDB v3](https://zenodo.org/records/18660057), 16.02.2026,
  175 166 значений; [статья](https://doi.org/10.1038/s41597-026-07047-z).
- [BigSolDB 2.2](https://zenodo.org/records/22648301), 07.09.2026,
  125 752 значения; [статья](https://doi.org/10.1038/s41597-025-05559-8).

Номера выше служат входом в семейство версий. **По умолчанию при каждом запуске
ноутбук спрашивает Zenodo о последней опубликованной версии**; результат может
отличаться от указанных версий. Для воспроизведения закрепите номера записей
в `PIN_RECORDS`. Версия набора BigSolDB «2.2» и индекс версии Zenodo «v3» — разные величины.

## Запуск
1. Поместить ноутбуки `00` и `01` в одну рабочую папку проекта.
2. При необходимости установить pandas; доступ к интернету и Zenodo обязателен.
3. Выполнить **Restart Kernel → Run All**.
4. Перенести напечатанные в конце пути в `CFG` ноутбука `01`.

Скачиваются только выбранные файлы двух семейств данных. По умолчанию также
загружаются справочники MixtureSolDB и сведения BigSolDB о методах, твёрдых формах
и плотностях. Производный файл gamma не нужен для подготовки и отключён.

In [ ]:
# Выполнить при необходимости, затем перезапустить kernel:
# %pip install pandas

## 1. Параметры

Данные разных версий сохраняются в отдельных каталогах:
`data/raw/<dataset>/<record_id>/`. Старые версии не заменяются новыми.
Повторный запуск проверяет уже скачанный файл по checksum Zenodo и использует его повторно.
Если содержимое записи Zenodo изменилось, файл обновляется только после успешной
проверки новой загрузки; для строгого контроля байтов можно задать `EXPECTED_SHA256`.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import time
import platform
import urllib.request
import urllib.error
from urllib.parse import urlparse, quote
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

RAW_DIR = Path("data/raw")
DOWNLOAD_AUXILIARY = True
DOWNLOAD_GAMMA = False
TIMEOUT_SECONDS = 60
MAX_ATTEMPTS = 3
USER_AGENT = "BinaryMixtureSolubilityResearch/1.0 (public Zenodo dataset download)"

# Пустой словарь = получить последнюю версию каждого семейства.
# Для воспроизведения, например:
# PIN_RECORDS = {"MixtureSolDB": "18660057", "BigSolDB": "22648301"}
PIN_RECORDS = {}

# Дополнительная фиксация SHA-256 ранее полученных основных CSV (необязательно).
# Скопируйте значения из download_manifest.json; другой файл вызовет ошибку.
# EXPECTED_SHA256 = {"MixtureSolDB": "...", "BigSolDB": "..."}
EXPECTED_SHA256 = {}

DATASETS = {
    "MixtureSolDB": {"seed_record_id": "18660057", "concept_record_id": "17846306",
                     "main_pattern": r"MixtureSolDB\.csv",
                     "required_columns": ["SMILES_Solute", "SMILES_Solvent1", "SMILES_Solvent2",
                        "Temperature_K", "Solubility(mole_fraction)", "Fraction_Solvent1",
                        "Fraction_Solvent2", "Fraction_Type"]},
    "BigSolDB": {"seed_record_id": "22648301", "concept_record_id": "15094978",
                 "main_pattern": r"BigSolDB(?:v[0-9]+(?:\.[0-9]+)*)?\.csv",
                 "required_columns": ["SMILES_Solute", "SMILES_Solvent", "Temperature_K",
                                       "Solubility(mole_fraction)"]},
}
unknown = (set(PIN_RECORDS) | set(EXPECTED_SHA256)) - set(DATASETS)
if unknown:
    raise ValueError(f"Неизвестные базы в параметрах: {unknown}")
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Каталог данных:", RAW_DIR.resolve())

Каталог данных: /Users/georg/projects/Solubility/Article_code/data/raw


## 2. Метаданные и выбор последней версии

Используются API записи Zenodo и ссылка `links.latest`, возвращённая самим сервером.
Проверяется принадлежность записи ожидаемому семейству (`conceptrecid`).
При ошибке API нет молчаливого перехода к старой версии: исправьте соединение
либо явно задайте `PIN_RECORDS`.

Документация: [Zenodo Records API](https://developers.zenodo.org/#records).

In [2]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def check_zenodo_url(url):
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname != "zenodo.org":
        raise ValueError(f"Ожидалась HTTPS-ссылка Zenodo: {url}")
    return url

def get_json(url):
    check_zenodo_url(url)
    last_error = None
    for attempt in range(MAX_ATTEMPTS):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as response:
                return json.load(response)
        except (urllib.error.URLError, TimeoutError, OSError, ValueError) as error:
            last_error = error
            if attempt + 1 < MAX_ATTEMPTS:
                delay = min(5 * (attempt + 1), 15)
                print(f"Метаданные: повтор {attempt + 2}/{MAX_ATTEMPTS} через {delay} с")
                time.sleep(delay)
    raise RuntimeError(f"Не удалось получить метаданные {url}: {last_error}") from last_error

def resolve_record(name, spec):
    pinned = name in PIN_RECORDS
    record_id = str(PIN_RECORDS.get(name, spec["seed_record_id"]))
    if not record_id.isdigit():
        raise ValueError("Номер записи Zenodo должен состоять из цифр")
    record = get_json(f"https://zenodo.org/api/records/{record_id}")
    if not pinned:
        latest_url = record.get("links", {}).get("latest")
        if not latest_url:
            raise ValueError(f"{name}: Zenodo не вернул links.latest. Закрепите запись явно.")
        record = get_json(latest_url)
    family = str(record.get("conceptrecid") or record.get("parent", {}).get("id", ""))
    if family != spec["concept_record_id"]:
        raise ValueError(f"{name}: чужое семейство Zenodo {family}, ожидалось {spec['concept_record_id']}")
    if not isinstance(record.get("files"), list) or not record["files"]:
        raise ValueError(f"{name}: список файлов отсутствует / схема API изменилась")
    return record

records = {name: resolve_record(name, spec) for name, spec in DATASETS.items()}
record_summary = []
for name, record in records.items():
    meta = record["metadata"]
    versions = record.get("versions", {})
    relations = meta.get("relations", {}).get("version", [])
    # В legacy API Zenodo index в relations.version начинается с 0.
    legacy_index = relations[0].get("index") if relations else None
    number = (int(legacy_index) + 1) if legacy_index is not None else None
    release = re.search(r"BigSolDB\s+([0-9]+(?:\.[0-9]+)+)", meta.get("title", ""))
    record_summary.append({"dataset": name, "record_id": str(record["id"]),
        "title": meta.get("title"), "version_label": meta.get("version") or (release.group(1) if release else None),
        "zenodo_version_number": number or versions.get("index"), "publication_date": meta.get("publication_date"),
        "doi": record.get("doi"), "selection": "pinned" if name in PIN_RECORDS else "latest"})
display(pd.DataFrame(record_summary))

,dataset,record_id,title,version_label,zenodo_version_number,publication_date,doi,selection
0,MixtureSolDB,18660057,"MixtureSolDB, dataset of solubility values for...",None,3,2026-02-16,10.5281/zenodo.18660057,latest
1,BigSolDB,22648301,BigSolDB 2.2: A Comprehensive Dataset of Solub...,2.2,3,2026-09-07,10.5281/zenodo.22648301,latest


## 3. Выбор файлов

Основная таблица BigSolDB выбирается по полному имени, чтобы случайно не взять
`gamma`, `densities`, `methods` или `solid_forms` вместо экспериментальной растворимости.
Если новая версия изменила имя основной таблицы, ноутбук остановится и покажет список файлов.

In [3]:
def choose_files(name, record, spec):
    files = record["files"]
    main = [f for f in files if re.fullmatch(spec["main_pattern"], f["key"], flags=re.IGNORECASE)]
    if len(main) != 1:
        raise ValueError(f"{name}: требуется ровно одна основная CSV. Найдено {len(main)}; файлы: {[f['key'] for f in files]}")
    selected = [main[0]]
    for f in files:
        key = f["key"].lower()
        if key == main[0]["key"].lower():
            continue
        auxiliary = (name == "MixtureSolDB" and key in {"solvent_abbreviation_legend.csv", "solvent_boiling_point.csv"})
        auxiliary |= (name == "BigSolDB" and any(key.endswith(s) for s in ["_methods.csv", "_solid_forms.csv", "_densities.csv"]))
        if (DOWNLOAD_AUXILIARY and auxiliary) or (DOWNLOAD_GAMMA and name == "BigSolDB" and key.endswith("_gamma.csv")):
            selected.append(f)
    return main[0]["key"], selected

selection = {name: choose_files(name, records[name], spec) for name, spec in DATASETS.items()}
plan = [{"dataset": name, "record_id": records[name]["id"], "file": f["key"],
         "size_MB": round(f.get("size", 0) / 1_000_000, 3), "checksum": f.get("checksum")}
        for name, (_, files) in selection.items() for f in files]
display(pd.DataFrame(plan))
print("Общий размер выбранных файлов:", round(sum(f.get("size", 0) for _, files in selection.values() for f in files) / 1_000_000, 1), "MB")

,dataset,record_id,file,size_MB,checksum
0,MixtureSolDB,18660057,MixtureSolDB.csv,41.108,md5:597e31685fe36c78d568f3d37b6e1336
1,MixtureSolDB,18660057,solvent_abbreviation_legend.csv,0.000,md5:afc090e75d79fd87e8ddb3648568b187
2,MixtureSolDB,18660057,solvent_boiling_point.csv,0.014,md5:057d5246d0b004f23e6f6d60b438aee1
3,BigSolDB,22648301,BigSolDBv2.2.csv,24.678,md5:a1e12ae9575319ac4f89fea6a63a6422
4,BigSolDB,22648301,BigSolDBv2.2_methods.csv,0.164,md5:4e7a688055d0e8ebf4c8c3a202b29b7c
5,BigSolDB,22648301,BigSolDBv2.2_solid_forms.csv,0.429,md5:3a077abad5373f52341f7915a9a38757
6,BigSolDB,22648301,BigSolDBv2.2_densities.csv,0.120,md5:059ae2deadb2ad1523fd4e24b2c88133


Общий размер выбранных файлов: 66.5 MB


## 4. Скачивание и контроль целостности

Загрузка потоковая: сначала `.part`, затем проверка размера и checksum Zenodo,
после чего атомарное переименование. Дополнительно вычисляется SHA-256.
Существующий корректный файл используется без повторной загрузки.
Файлы со схемой CSV проверяются на следующем этапе.

In [4]:
def digest_file(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def checksum_valid(path, descriptor):
    if not path.is_file():
        return False
    expected_size = descriptor.get("size")
    if expected_size is not None and path.stat().st_size != int(expected_size):
        return False
    checksum = descriptor.get("checksum", "")
    if ":" not in checksum:
        raise ValueError(f"Нет проверяемой checksum у {descriptor['key']}")
    algorithm, value = checksum.split(":", 1)
    return digest_file(path, algorithm) == value.lower()

def download_file(record, descriptor, destination, expected_sha256=None):
    key = descriptor["key"]
    if Path(key).name != key or "/" in key or "\\" in key:
        raise ValueError(f"Недопустимое имя файла: {key}")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / key
    if checksum_valid(target, descriptor):
        sha = digest_file(target)
        if expected_sha256 and sha != expected_sha256.lower():
            raise ValueError(f"SHA-256 не совпадает с закреплённым для {key}")
        print("Уже проверен:", target)
        return target, sha, "reused"
    links = descriptor.get("links", {})
    url = links.get("content") or links.get("self")
    if not url:
        url = f"https://zenodo.org/records/{record['id']}/files/{quote(key, safe='')}?download=1"
    check_zenodo_url(url)
    partial = target.with_name(target.name + ".part")
    last_error = None
    for attempt in range(MAX_ATTEMPTS):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as response, open(partial, "wb") as output:
                copied, next_report = 0, 10_000_000
                while chunk := response.read(1024 * 1024):
                    output.write(chunk)
                    copied += len(chunk)
                    if copied >= next_report:
                        print(f"  {key}: {copied / 1_000_000:.1f} MB")
                        next_report += 10_000_000
            if not checksum_valid(partial, descriptor):
                raise ValueError(f"Размер / checksum не совпали для {key}")
            sha = digest_file(partial)
            if expected_sha256 and sha != expected_sha256.lower():
                raise ValueError(f"SHA-256 не совпадает с закреплённым для {key}")
            partial.replace(target)
            print("Скачан и проверен:", target)
            return target, sha, "downloaded"
        except (urllib.error.URLError, TimeoutError, OSError, ValueError) as error:
            last_error = error
            partial.unlink(missing_ok=True)
            if expected_sha256 and isinstance(error, ValueError) and "закреплённым" in str(error):
                raise
            if attempt + 1 < MAX_ATTEMPTS:
                time.sleep(min(5 * (attempt + 1), 15))
                print(f"Повтор {attempt + 2}/{MAX_ATTEMPTS}: {key}")
    raise RuntimeError(f"Не удалось скачать {url}: {last_error}") from last_error

downloaded, main_paths = [], {}
for name, (main_key, files) in selection.items():
    record = records[name]
    destination = RAW_DIR / name / str(record["id"])
    for descriptor in files:
        path, sha, action = download_file(record, descriptor, destination,
            EXPECTED_SHA256.get(name) if descriptor["key"] == main_key else None)
        downloaded.append({"dataset": name, "record_id": str(record["id"]), "filename": descriptor["key"],
                           "path": str(path), "size_bytes": path.stat().st_size,
                           "zenodo_checksum": descriptor["checksum"], "sha256": sha, "action": action})
        if descriptor["key"] == main_key:
            main_paths[name] = path
    (destination / "zenodo_record.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(downloaded)[["dataset", "filename", "size_bytes", "action"]])

  MixtureSolDB.csv: 10.5 MB
  MixtureSolDB.csv: 21.0 MB
  MixtureSolDB.csv: 30.4 MB
  MixtureSolDB.csv: 40.9 MB
Скачан и проверен: data/raw/MixtureSolDB/18660057/MixtureSolDB.csv
Скачан и проверен: data/raw/MixtureSolDB/18660057/solvent_abbreviation_legend.csv
Скачан и проверен: data/raw/MixtureSolDB/18660057/solvent_boiling_point.csv
  BigSolDBv2.2.csv: 10.5 MB
  BigSolDBv2.2.csv: 21.0 MB
Скачан и проверен: data/raw/BigSolDB/22648301/BigSolDBv2.2.csv
Скачан и проверен: data/raw/BigSolDB/22648301/BigSolDBv2.2_methods.csv
Скачан и проверен: data/raw/BigSolDB/22648301/BigSolDBv2.2_solid_forms.csv
Скачан и проверен: data/raw/BigSolDB/22648301/BigSolDBv2.2_densities.csv


,dataset,filename,size_bytes,action
0,MixtureSolDB,MixtureSolDB.csv,41107737,downloaded
1,MixtureSolDB,solvent_abbreviation_legend.csv,459,downloaded
2,MixtureSolDB,solvent_boiling_point.csv,14209,downloaded
3,BigSolDB,BigSolDBv2.2.csv,24678209,downloaded
4,BigSolDB,BigSolDBv2.2_methods.csv,164212,downloaded
5,BigSolDB,BigSolDBv2.2_solid_forms.csv,429267,downloaded
6,BigSolDB,BigSolDBv2.2_densities.csv,120029,downloaded


## 5. Проверка основной CSV

Проверяются обязательные колонки для ноутбука `01`, успешное чтение всей таблицы,
число строк, типы долей MixtureSolDB. Новые дополнительные колонки сохраняются.
Используется `Solubility(mole_fraction)`, а не концентрация mol/L.

In [5]:
def inspect_csv(path, required_columns):
    header = pd.read_csv(path, nrows=0)
    missing = sorted(set(required_columns) - set(header.columns))
    if missing:
        raise ValueError(f"{path.name}: несовместимая схема, отсутствуют {missing}. Обновите mapping в 01.")
    n_rows = sum(len(chunk) for chunk in pd.read_csv(path, chunksize=100_000, low_memory=False))
    if n_rows == 0:
        raise ValueError(f"Пустая таблица: {path}")
    return {"path": str(path), "rows": n_rows, "columns": list(header.columns)}

csv_info = {name: inspect_csv(path, DATASETS[name]["required_columns"]) for name, path in main_paths.items()}
display(pd.DataFrame([{"dataset": name, "rows": info["rows"], "n_columns": len(info["columns"])}
                      for name, info in csv_info.items()]))
for name, path in main_paths.items():
    print(name, "—", path.name)
    display(pd.read_csv(path, nrows=3))
fraction_counts = pd.read_csv(main_paths["MixtureSolDB"], usecols=["Fraction_Type"])["Fraction_Type"].value_counts(dropna=False)
display(fraction_counts.rename("n").to_frame())

,dataset,rows,n_columns
0,MixtureSolDB,175166,19
1,BigSolDB,125752,14


MixtureSolDB — MixtureSolDB.csv


,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solubility(g/100g),LogS(g/100g),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,Fraction_Solvent1,Fraction_Solvent2,Fraction_Type,Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,IsPureSolventEndpoint
0,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,212.339615,2.327031,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078,True,10.1007/s10973-018-7684-y,True
1,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,242.315737,2.384382,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078,True,10.1007/s10973-018-7684-y,True
2,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,282.038981,2.450309,acetonitrile,water,CC#N,O,0.0,1.0,mole,Rivastigmine tartrate,129101-54-8,6918078,True,10.1007/s10973-018-7684-y,True


BigSolDB — BigSolDBv2.2.csv


,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),LogS(mole_fraction),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,solid_form_label
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,-3.221849,Ethylene glycol monoeicosate,26158-80-5,538813,No,10.1007/bf00649573,NaN
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,-2.920819,Ethylene glycol monoeicosate,26158-80-5,538813,No,10.1007/bf00649573,NaN
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,-2.698970,Ethylene glycol monoeicosate,26158-80-5,538813,No,10.1007/bf00649573,NaN


,n
Fraction_Type,
mass,91463
mole,83703


## 6. Манифест и передача путей в ноутбук 01

Записывается дата загрузки, выбор latest/pinned, DOI, номера записей и семейств,
полные метаданные Zenodo, проверки CSV, MD5 и SHA-256.
Последний успешный запуск отражён в `download_manifest.json`; снимок манифеста
также сохраняется в каталоге `manifests/`.

**Последние версии полезны для нового исследования, но после выбора протокола
следует закрепить номера записей и SHA-256 перед сравнением моделей.**
Сами авторы могут обновить файлы в записи, поэтому одного record_id недостаточно
для проверки идентичности байтов.

In [6]:
manifest = {
    "notebook_protocol": "download-1.0.0", "created_at_utc": utc_now(),
    "configuration": {"pin_records": PIN_RECORDS, "expected_sha256": EXPECTED_SHA256,
                      "download_auxiliary": DOWNLOAD_AUXILIARY, "download_gamma": DOWNLOAD_GAMMA},
    "records": [{**row, "concept_record_id": DATASETS[row["dataset"]]["concept_record_id"],
                 "metadata_path": str(RAW_DIR / row["dataset"] / row["record_id"] / "zenodo_record.json")}
                for row in record_summary],
    "files": downloaded, "csv_inspection": csv_info,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__},
}
payload = json.dumps(manifest, ensure_ascii=False, indent=2)
(RAW_DIR / "download_manifest.json").write_text(payload, encoding="utf-8")
archive_dir = RAW_DIR / "manifests"
archive_dir.mkdir(exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
(archive_dir / f"download-{stamp}.json").write_text(payload, encoding="utf-8")
preprocessing_inputs = {"MIXTURE_PATH": str(main_paths["MixtureSolDB"]),
                        "BIGSOLDB_PATH": str(main_paths["BigSolDB"]), "USE_BIGSOLDB": True}
(RAW_DIR / "preprocessing_inputs.json").write_text(json.dumps(preprocessing_inputs, ensure_ascii=False, indent=2), encoding="utf-8")
print("Готово. В ячейке параметров ноутбука 01, после определения CFG, добавьте:")
print("CFG.update(" + repr(preprocessing_inputs) + ")")
print("\nИли загрузите сохранённые пути (если рабочая папка та же):")
print(f"CFG.update(json.loads(Path({str(RAW_DIR / 'preprocessing_inputs.json')!r}).read_text(encoding='utf-8')))")
print("\nЧтобы закрепить версии в этом ноутбуке:")
print("PIN_RECORDS =", {name: str(record["id"]) for name, record in records.items()})
print("EXPECTED_SHA256 =", {name: next(f["sha256"] for f in downloaded if f["dataset"] == name and f["path"] == str(path))
                             for name, path in main_paths.items()})

Готово. В ячейке параметров ноутбука 01, после определения CFG, добавьте:
CFG.update({'MIXTURE_PATH': 'data/raw/MixtureSolDB/18660057/MixtureSolDB.csv', 'BIGSOLDB_PATH': 'data/raw/BigSolDB/22648301/BigSolDBv2.2.csv', 'USE_BIGSOLDB': True})

Или загрузите сохранённые пути (если рабочая папка та же):
CFG.update(json.loads(Path('data/raw/preprocessing_inputs.json').read_text(encoding='utf-8')))

Чтобы закрепить версии в этом ноутбуке:
PIN_RECORDS = {'MixtureSolDB': '18660057', 'BigSolDB': '22648301'}
EXPECTED_SHA256 = {'MixtureSolDB': 'be72940b1d9ba03eacf142cfb26e774b658617dbf8fefb3cfb96f27c4da5e51d', 'BigSolDB': '6b63f86601369f92530bedf6b93c3fb92a2731390b28a0aabf55ad005acba401'}


## Что дальше

Выполнить ноутбук `01` с полученными путями. Колонки основных CSV текущих версий
совместимы с его mapping. Дополнительные данные BigSolDB о методах и твёрдых формах
пока сохраняются отдельно: их включение в правила выбора endpoints — отдельное
методическое решение, которое нужно согласовать до обучения.

Для публикации проекта не забудьте сослаться на статьи и конкретные DOI версий баз,
использованных в расчёте. Полные метаданные лицензий и авторов доступны в сохранённых
`zenodo_record.json`.